In [8]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import gymnasium as gym
from IPython.display import Video

In [9]:
class ActorPolicyNetwork(nn.Module):
    def __init__(self,
                 input_state_features=8, 
                 num_actions=4,
                 hidden_features=128):
        
        super(ActorPolicyNetwork, self).__init__()

        self.fc1 = nn.Linear(input_state_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        pi = F.softmax(self.fc3(x), dim=-1)
        return pi

class CriticValueNetwork(nn.Module):
    def __init__(self, input_state_features=8, hidden_features=128):
        super(CriticValueNetwork, self).__init__()
        self.fc1 = nn.Linear(input_state_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, 1)  

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        value = self.fc3(x)
        return value
        

In [35]:
### In actor-critic architectures we use bootstrapping to bootstrap the value of our current state, action pair
# recall Q(s,a) = r_t+1 + gamma*V(s')
# recall A(s,a) = r_t+1 + gamma*V(s') - V(s)

### we don't need a compute returns function anymore since we will be computing returns on the fly
# with actor-critic we use bootstrapped estimates to update our policy function

def train(
    env,
    gamma=0.95,
    episodes=1000,
    learning_rate=3e-4,
    input_state_features=8, 
    num_actions=4,
    hidden_features=128,
    device='mps', 
    entropy_weight=0.9,
    log_freq=50,
    running_avg_steps = 50,
):
    # initialize networks
    policy_network = ActorPolicyNetwork(
        input_state_features=input_state_features, 
        num_actions=num_actions,
        hidden_features=hidden_features
    ).to(device)
    
    value_network = CriticValueNetwork(
        input_state_features=input_state_features, 
        hidden_features=hidden_features
    ).to(device)

    optimizer_policy = optim.Adam(policy_network.parameters(), lr=learning_rate)
    optimizer_value = optim.Adam(value_network.parameters(), lr=learning_rate)
    
    log = {
        "scores" : [],
        "running_avg_scores" : []
    }

    for i in range(episodes):
        
        state, _ = env.reset()

        entropies = []
        done = False
        total_rewards = 0
    
        while not done: 

            # state is an array of 8 floats in LunarLander
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            
            possible_actions = policy_network(state_tensor)
            action = torch.multinomial(possible_actions, num_samples=1).item() # returns integer
            
            # remember whenever we log or divide, we add 1e-8 to prevent dividing by zero
            entropy = -torch.sum(possible_actions*torch.log(possible_actions + 1e-8)) # entropy is -logprobs of the action * probs of the action
            entropies.append(entropy)

            log_probs = torch.log(possible_actions[0, action]) # to compute loss later
            
            next_state, reward, terminal, truncated, _ = env.step(action)
            done = truncated or terminal

            next_state_tensor = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0).to(device)
            
            if not done: 
                value_next_state = value_network(next_state_tensor).squeeze(dim=0)
            else:
                torch.Tensor([[0.0]])# output is (1,1) so we need to fix that

            value_state = value_network(state_tensor).squeeze(dim=0) # output is (1,1) so we need to fix that

            td_target = reward + gamma*value_next_state.detach()
            td_error = td_target - value_state.detach() # we treat the value as a scalar, it is not getting updated here

            # loss = - logprobs of action taken * bootstrapped returns (return + gamma*value - current value of next state) <- TD error
            policy_loss = -log_probs*td_error - entropy_weight * entropy

            # value function = value the model would have predicted vs observed value of the state (TD target) - again using the Q value as an estimator
            value_loss = F.smooth_l1_loss(value_state, td_target)

            optimizer_policy.zero_grad()
            optimizer_value.zero_grad()

            policy_loss.backward()
            value_loss.backward()

            torch.nn.utils.clip_grad_norm(policy_network.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1
            torch.nn.utils.clip_grad_norm(value_network.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1
            
            optimizer_policy.step()
            optimizer_value.step()

            state = next_state
            total_rewards += reward

        log["scores"].append(total_rewards)
        running_avg_scores = np.mean(log["scores"][-running_avg_steps:])
        log["running_avg_scores"].append(running_avg_scores)

        if i % log_freq == 0:
            print(f"Episode {i}, Total Reward: {total_rewards}, Average Reward: {running_avg_scores}")

        if running_avg_scores >= 200: 
            print("Training complete")
            break
        

    return policy_network, log

In [36]:
### Play Game ###
env = gym.make("LunarLander-v3", render_mode="rgb_array")
policy, log = train(env, device="mps")

/var/folders/jl/ppm07cc55jgfvx0gzqfyhtrw0000gn/T/ipykernel_86808/2210963142.py:90: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(policy_network.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1
/var/folders/jl/ppm07cc55jgfvx0gzqfyhtrw0000gn/T/ipykernel_86808/2210963142.py:91: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(value_network.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1


Episode 0, Total Reward: -113.3638203426041, Average Reward: -113.3638203426041


KeyboardInterrupt: 